# Exp7.3.5 — Hidden-state information loss across LIF quantization

Analysis-only notebook for the frozen Exp7.3 A2 hidden-state probes. The primary comparison is pre-threshold membrane versus binary spike at L1 and L2 under whole-sequence and ordered Fixed250 readouts.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
while not (ROOT / 'pyproject.toml').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
ART = ROOT / 'notebooks' / 'artifacts' / 'experiment_7_3_5_hidden_state_information_loss' / 'hidden_state_information_loss_v1'
manifest = json.loads((ART / 'manifest.json').read_text())
method_runs = pd.read_csv(ART / 'method_runs.csv')
method_summary = pd.read_csv(ART / 'method_summary.csv')
contrast_runs = pd.read_csv(ART / 'contrast_runs.csv')
contrast_summary = pd.read_csv(ART / 'contrast_summary.csv')
source_checks = pd.read_csv(ART / 'source_reproduction_checks.csv')
manifest

## Primary method summary
The three primary hidden states are synaptic current, pre-reset membrane, and binary spike. Post-reset membrane is retained only as a secondary reset diagnostic.

In [ ]:
primary = method_summary[method_summary['state_role'] == 'primary'].copy()
cols = ['aggregation', 'layer', 'state', 'test_ba_mean', 'test_ba_std', 'val_ba_mean', 'selected_C_mean']
primary[cols].sort_values(['aggregation', 'layer', 'state'])

## Whole-sequence information
Within each layer, compare `syn_current -> pre_reset -> spike`. A large drop from pre-reset membrane to spike indicates hidden binary quantization loss even without an output spiking neuron.

In [ ]:
state_order = ['syn_current', 'pre_reset', 'spike']
for layer in ['l1', 'l2']:
    frame = primary[(primary['aggregation'] == 'whole_mean') & (primary['layer'] == layer)].set_index('state').loc[state_order]
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.errorbar(state_order, 100 * frame['test_ba_mean'], yerr=100 * frame['test_ba_std'], marker='o', capsize=4)
    ax.set_ylabel('Test balanced accuracy (%)')
    ax.set_title(f'{layer.upper()} whole-sequence mean')
    ax.grid(axis='y', alpha=0.25)
    plt.show()

## Ordered Fixed250 temporal information
This view retains absolute 250 ms bin order. A larger membrane-to-spike gap here than in whole-mean indicates preferential loss of temporal/phase information.

In [ ]:
for layer in ['l1', 'l2']:
    frame = primary[(primary['aggregation'] == 'fixed250_ordered_mean') & (primary['layer'] == layer)].set_index('state').loc[state_order]
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.errorbar(state_order, 100 * frame['test_ba_mean'], yerr=100 * frame['test_ba_std'], marker='o', capsize=4)
    ax.set_ylabel('Test balanced accuracy (%)')
    ax.set_title(f'{layer.upper()} ordered Fixed250 mean')
    ax.grid(axis='y', alpha=0.25)
    plt.show()

## Primary information-loss contrasts
Positive `spike_quantization_pre_minus_spike` means the pre-threshold membrane retains more linearly accessible information than the binary spike. Positive `membrane_integration_pre_minus_current` means membrane integration improves accessibility relative to synaptic current.

In [ ]:
primary_contrasts = contrast_summary[contrast_summary['contrast'].isin(manifest['primary_contrasts'])].copy()
primary_contrasts.sort_values(['aggregation', 'layer', 'contrast'])

In [ ]:
quant = primary_contrasts[primary_contrasts['contrast'] == 'spike_quantization_pre_minus_spike'].copy()
for aggregation in ['whole_mean', 'fixed250_ordered_mean']:
    frame = quant[quant['aggregation'] == aggregation].set_index('layer').loc[['l1', 'l2']]
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.bar(['L1', 'L2'], frame['mean'], yerr=frame['std'], capsize=4)
    ax.axhline(0, linewidth=1)
    ax.set_ylabel('Pre-reset minus spike test BA (pp)')
    ax.set_title(aggregation)
    ax.grid(axis='y', alpha=0.25)
    plt.show()

## Secondary reset diagnostics

In [ ]:
secondary = contrast_summary[contrast_summary['contrast'].isin(manifest['secondary_contrasts'])].copy()
secondary.sort_values(['aggregation', 'layer', 'contrast'])

## Source replay audit
All rows should stay within the manifest's extraction tolerance. This verifies that the diagnostic replay uses the same L2 binary-spike realization as the frozen Exp7.3 A2 cache.

In [ ]:
source_checks.sort_values(['seed', 'split', 'aggregation'])